# Mais sobre `SELECT`

- temporary tables
- views
- variables
- subqueries


In [74]:
from functools import partial
from dotenv import load_dotenv
import insperautograder.jupyter as ia
import mysql.connector
import os

load_dotenv(override=True)

connection = mysql.connector.connect(
    host=os.getenv("MD_DB_SERVER"),
    user=os.getenv("MD_DB_USERNAME"),
    password=os.getenv("MD_DB_PASSWORD"),
    port=int(os.getenv("MD_DB_PORT", 3306)),
    database="sakila"
)

def run_db_query(connection, query, args=None):
    with connection.cursor() as cursor:
        print("Executando query:")
        for result in cursor.execute(query, multi=True):
            if result.with_rows:
                for row in result.fetchall():
                    print(row)
            else:
                print(f"{result.rowcount} linhas afetadas.")

db = partial(run_db_query, connection)

## Exercícios para entrega

Esta aula tem atividade para entrega, confira os prazos e exercícios

In [75]:
ia.tasks()

|    | Atividade    | De                  | Até                 | Conta como ATV?   | % Nota Atraso   |
|---:|:-------------|:--------------------|:--------------------|:------------------|:----------------|
|  0 | newborn      | 2026-02-09 00:00:00 | 2026-05-30 00:00:00 | Não               | 0%              |
|  1 | select01     | 2026-02-12 16:00:00 | 2026-02-22 23:59:59 | Sim               | 25%             |
|  2 | ddl          | 2026-02-26 12:00:00 | 2026-03-05 23:59:59 | Sim               | 25%             |
|  3 | dml          | 2026-03-02 14:15:00 | 2026-03-08 23:59:59 | Sim               | 25%             |
|  4 | agg_join     | 2026-03-05 14:15:00 | 2026-03-12 23:59:59 | Sim               | 25%             |
|  5 | group_having | 2026-03-09 14:15:00 | 2026-03-15 23:59:59 | Sim               | 25%             |
|  6 | views        | 2026-03-12 14:15:00 | 2026-03-19 23:59:59 | Sim               | 25%             |
|  7 | sql_review1  | 2026-03-16 14:15:00 | 2026-03-22 23:59:59 | Sim               | 25%             |

In [76]:
ia.grades(by="task")

|    | Tarefa       |   Nota | Conta como ATV?   |
|---:|:-------------|-------:|:------------------|
|  0 | newborn      |  10    | Não               |
|  1 | select01     |  10    | Sim               |
|  2 | ddl          |  10    | Sim               |
|  3 | dml          |  10    | Sim               |
|  4 | agg_join     |  10    | Sim               |
|  5 | group_having |  10    | Sim               |
|  6 | views        |   9.17 | Sim               |
|  7 | sql_review1  |   0    | Sim               |

In [77]:
ia.grades(task="views")

|    | Atividade   | Exercício   |   Peso |   Nota |   Nota Sem Atraso |   Nota Com Atraso |
|---:|:------------|:------------|-------:|-------:|------------------:|------------------:|
|  0 | views       | ex01        |      1 |     10 |                10 |                 0 |
|  1 | views       | ex02        |      1 |     10 |                10 |                 0 |
|  2 | views       | ex03        |      1 |     10 |                10 |                 0 |
|  3 | views       | ex04        |      1 |     10 |                10 |                 0 |
|  4 | views       | ex05        |      1 |     10 |                10 |                 0 |
|  5 | views       | ex06        |      1 |     10 |                10 |                 0 |
|  6 | views       | ex07        |      1 |     10 |                10 |                 0 |
|  7 | views       | ex08        |      1 |     10 |                10 |                 0 |
|  8 | views       | ex09        |      1 |     10 |                10 |                 0 |
|  9 | views       | ex10        |      1 |     10 |                10 |                 0 |
| 10 | views       | ex11        |      1 |     10 |                10 |                 0 |
| 11 | views       | ex12        |      1 |      0 |                 0 |                 0 |

In [78]:
# Média de ATV, dividindo por n-2
ia.average(excluded_count=2)

|    |   Média de ATV |
|---:|---------------:|
|  0 |             10 |

## Aquecimento

Quanta receita foi gerada para cada categoria de filmes? Mostre o nome da categoria e a receita. Ordene da maior receita para para a menor.

In [79]:
sql_ex01 = """
SELECT 
    c.name, SUM(p.amount)
FROM
    category c
        JOIN
    film_category fc USING (category_id)
        JOIN
    film f USING (film_id)
        JOIN
    inventory i USING (film_id)
        JOIN
    rental r USING (inventory_id)
        JOIN
    payment p USING (rental_id)
GROUP BY c.name
ORDER BY SUM(p.amount) DESC
"""

db(sql_ex01)

Executando query:
('Sports', Decimal('5314.21'))
('Sci-Fi', Decimal('4756.98'))
('Animation', Decimal('4656.30'))
('Drama', Decimal('4587.39'))
('Comedy', Decimal('4383.58'))
('Action', Decimal('4375.85'))
('New', Decimal('4351.62'))
('Games', Decimal('4281.33'))
('Foreign', Decimal('4270.67'))
('Family', Decimal('4226.07'))
('Documentary', Decimal('4217.52'))
('Horror', Decimal('3722.54'))
('Children', Decimal('3655.55'))
('Classics', Decimal('3639.59'))
('Travel', Decimal('3549.64'))
('Music', Decimal('3417.72'))


In [80]:
ia.sender(answer="sql_ex01", task="views", question="ex01", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex01', style=ButtonStyle()), Output()), _dom_classes=('widget…

Liste os 10 filmes mais alugados e o numero de vezes em que foram alugados. Ordene do mais alugado para o menos alugado.

Caso exista empate, como segundo critério, ordene de forma crescente pelo nome do filme.

In [81]:
sql_ex02 = """
SELECT 
    f.title, COUNT(r.rental_id)
FROM
    film f
        JOIN
    inventory i USING (film_id)
        JOIN
    rental r USING (inventory_id)
GROUP BY film_id
ORDER BY COUNT(r.rental_id) DESC, f.title ASC
LIMIT 10
"""

db(sql_ex02)

Executando query:
('BUCKET BROTHERHOOD', 34)
('ROCKETEER MOTHER', 33)
('FORWARD TEMPLE', 32)
('GRIT CLOCKWORK', 32)
('JUGGLER HARDLY', 32)
('RIDGEMONT SUBMARINE', 32)
('SCALAWAG DUCK', 32)
('APACHE DIVINE', 31)
('GOODFELLAS SALUTE', 31)
('HOBBIT ALIEN', 31)


In [82]:
ia.sender(answer="sql_ex02", task="views", question="ex02", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex02', style=ButtonStyle()), Output()), _dom_classes=('widget…

**DESAFIO:** Liste os filmes e o numero de vezes em que foram alugados apenas para filmes que foram alugados mais do que a média de numero de alugueis por filme. *Dica*: Serão necessários múltiplos SELECTs. Faça cada um em uma célula diferente.

In [83]:
db("""
-- SUA QUERY AQUI!
""")
db("""
-- SUA QUERY AQUI!
""")
db("""
-- SUA QUERY AQUI!
""")
db("""
-- SUA QUERY AQUI!
""")

Executando query:
0 linhas afetadas.
Executando query:
0 linhas afetadas.
Executando query:
0 linhas afetadas.
Executando query:
0 linhas afetadas.


## Views

Uma *view* é uma tabela virtual, construida a partir de um comando `SELECT`. Por exemplo: execute o código a seguir.

In [84]:
db("""
DROP VIEW IF EXISTS movie_count;
""")

db("""
CREATE VIEW movie_count AS
    SELECT 
        title, COUNT(rental_id) as cnt
    FROM
        film
        LEFT OUTER JOIN inventory USING (film_id)
        LEFT OUTER JOIN rental USING (inventory_id)
    GROUP BY
        film_id
    ORDER BY
        cnt ASC;
""")

Executando query:
0 linhas afetadas.
Executando query:
0 linhas afetadas.


Agora temos uma *view* chamada `movie_count`. Vamos verificar que ela funcionou, listando as 30 primeiras linhas:

In [85]:
db("""
SELECT 
    *
FROM
    movie_count 
LIMIT 
    30
""")

Executando query:
('ALICE FANTASIA', 0)
('APOLLO TEEN', 0)
('ARGONAUTS TOWN', 0)
('ARK RIDGEMONT', 0)
('ARSENIC INDEPENDENCE', 0)
('BOONDOCK BALLROOM', 0)
('BUTCH PANTHER', 0)
('CATCH AMISTAD', 0)
('CHINATOWN GLADIATOR', 0)
('CHOCOLATE DUCK', 0)
('COMMANDMENTS EXPRESS', 0)
('CROSSING DIVORCE', 0)
('CROWDS TELEMARK', 0)
('CRYSTAL BREAKING', 0)
('DAZED PUNK', 0)
('DELIVERANCE MULHOLLAND', 0)
('FIREHOUSE VIETNAM', 0)
('FLOATS GARDEN', 0)
('FRANKENSTEIN STRANGER', 0)
('GLADIATOR WESTWARD', 0)
('GUMP DATE', 0)
('HATE HANDICAP', 0)
('HOCUS FRIDA', 0)
('KENTUCKIAN GIANT', 0)
('KILL BROTHERHOOD', 0)
('MUPPET MILE', 0)
('ORDER BETRAYED', 0)
('PEARL DESTINY', 0)
('PERDITION FARGO', 0)
('PSYCHO SHRUNK', 0)


In [86]:
db("""
SELECT 
    * 
FROM 
    movie_count 
ORDER BY
    cnt DESC
LIMIT 30
""")

Executando query:
('BUCKET BROTHERHOOD', 34)
('ROCKETEER MOTHER', 33)
('JUGGLER HARDLY', 32)
('GRIT CLOCKWORK', 32)
('FORWARD TEMPLE', 32)
('SCALAWAG DUCK', 32)
('RIDGEMONT SUBMARINE', 32)
('ZORRO ARK', 31)
('WIFE TURN', 31)
('TIMBERLAND SKY', 31)
('RUSH GOODFELLAS', 31)
('ROBBERS JOON', 31)
('NETWORK PEAK', 31)
('HOBBIT ALIEN', 31)
('GOODFELLAS SALUTE', 31)
('APACHE DIVINE', 31)
('WITCHES PANIC', 30)
('SUSPECTS QUILLS', 30)
('SHOCK CABIN', 30)
('RUGRATS SHAKESPEARE', 30)
('PULP BEVERLY', 30)
('MUSCLE BRIGHT', 30)
('MASSACRE USUAL', 30)
('MARRIED GO', 30)
('IDOLS SNATCHERS', 30)
('HARRY IDAHO', 30)
('GRAFFITI LOVE', 30)
('FROST HEAD', 30)
('ENGLISH BULWORTH', 30)
('DOGMA FAMILY', 30)


Agora suponha que alteramos a tabela `film`, mudando o nome do filme "DAZED PUNK" para "STONED PUNK".

**Atividade**: Do it.

In [87]:
db("""
UPDATE film
SET title = 'STONED PUNK'
WHERE title = 'DAZED PUNK';
""")

Executando query:
1 linhas afetadas.


Verifique agora a nossa *view*:

In [88]:
db("""
SELECT
    *
FROM
    movie_count
LIMIT
    30
""")

Executando query:
('ALICE FANTASIA', 0)
('APOLLO TEEN', 0)
('ARGONAUTS TOWN', 0)
('ARK RIDGEMONT', 0)
('ARSENIC INDEPENDENCE', 0)
('BOONDOCK BALLROOM', 0)
('BUTCH PANTHER', 0)
('CATCH AMISTAD', 0)
('CHINATOWN GLADIATOR', 0)
('CHOCOLATE DUCK', 0)
('COMMANDMENTS EXPRESS', 0)
('CROSSING DIVORCE', 0)
('CROWDS TELEMARK', 0)
('CRYSTAL BREAKING', 0)
('STONED PUNK', 0)
('DELIVERANCE MULHOLLAND', 0)
('FIREHOUSE VIETNAM', 0)
('FLOATS GARDEN', 0)
('FRANKENSTEIN STRANGER', 0)
('GLADIATOR WESTWARD', 0)
('GUMP DATE', 0)
('HATE HANDICAP', 0)
('HOCUS FRIDA', 0)
('KENTUCKIAN GIANT', 0)
('KILL BROTHERHOOD', 0)
('MUPPET MILE', 0)
('ORDER BETRAYED', 0)
('PEARL DESTINY', 0)
('PERDITION FARGO', 0)
('PSYCHO SHRUNK', 0)


Como você pode ver, as views são tabelas virtuais que são **automaticamente atualizadas quando as tabelas originais são modificadas**.

Sempre que você realizar modificações nos dados, dê `commit` ou `rollback`. Ainda, evite executar múltiplas vezes as linhas de código que criem a conexão sem antes ter fechado a conexão ativa.

Vamos desfazer as alterações:

In [89]:
connection.rollback()

Conferindo:

In [90]:
db("""
SELECT
    *
FROM
    movie_count
LIMIT
    30
""")

Executando query:
('ALICE FANTASIA', 0)
('APOLLO TEEN', 0)
('ARGONAUTS TOWN', 0)
('ARK RIDGEMONT', 0)
('ARSENIC INDEPENDENCE', 0)
('BOONDOCK BALLROOM', 0)
('BUTCH PANTHER', 0)
('CATCH AMISTAD', 0)
('CHINATOWN GLADIATOR', 0)
('CHOCOLATE DUCK', 0)
('COMMANDMENTS EXPRESS', 0)
('CROSSING DIVORCE', 0)
('CROWDS TELEMARK', 0)
('CRYSTAL BREAKING', 0)
('DAZED PUNK', 0)
('DELIVERANCE MULHOLLAND', 0)
('FIREHOUSE VIETNAM', 0)
('FLOATS GARDEN', 0)
('FRANKENSTEIN STRANGER', 0)
('GLADIATOR WESTWARD', 0)
('GUMP DATE', 0)
('HATE HANDICAP', 0)
('HOCUS FRIDA', 0)
('KENTUCKIAN GIANT', 0)
('KILL BROTHERHOOD', 0)
('MUPPET MILE', 0)
('ORDER BETRAYED', 0)
('PEARL DESTINY', 0)
('PERDITION FARGO', 0)
('PSYCHO SHRUNK', 0)


### Vamos praticar

Verifique quantas vezes o filme "COWBOY DOOM" foi alugado usando a view `movie_count`

In [91]:
sql_ex03 = """
SELECT mv.cnt
FROM movie_count mv
WHERE mv.title = 'COWBOY DOOM'
"""

db(sql_ex03)

Executando query:
(8,)


In [92]:
ia.sender(answer="sql_ex03", task="views", question="ex03", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex03', style=ButtonStyle()), Output()), _dom_classes=('widget…

Registre, na loja id=1, um aluguel do filme `"COWBOY DOOM"` (com o menor id possível), feito pelo funcionario "Jon Stephens" para o cliente "JESSIE BANKS", na data '2019-01-01', com data de retorno '2019-01-08'.

**Dicas**:

- Utilize `DESCRIBE` para descobrir se o `rental_id` precisa ser informado ou se é autoincremento (se for autoincremento, ele já receberá automaticamente o menor id possível)ç
- Será necessário utilizar `SELECT` com `LIMIT 1` para buscar os **ids** do funcionário e inventário do filme.

In [93]:
sql_ex04 = """
INSERT INTO rental(rental_date, inventory_id,customer_id,return_date,staff_id)
VALUES(
		'2019-01-01',
		(SELECT i.inventory_id FROM inventory i JOIN film f USING(film_id) WHERE store_id = 1 AND f.title = 'COWBOY DOOM' LIMIT 1),
        (SELECT c.customer_id FROM customer c WHERE c.first_name = 'JESSIE' AND c.last_name = 'BANKS'LIMIT 1),
        '2019-01-08',
        (SELECT s.staff_id FROM staff s WHERE s.first_name = 'Jon' AND s.last_name = 'Stephens' LIMIT 1)
        )
"""

db(sql_ex04)

Executando query:
1 linhas afetadas.


In [94]:
db("""
-- SUA QUERY AQUI CASO NECESSITE DE MAIS!
-- Crie quantas forem necesárias!
""")

Executando query:
0 linhas afetadas.


In [95]:
ia.sender(answer="sql_ex04", task="views", question="ex04", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex04', style=ButtonStyle()), Output()), _dom_classes=('widget…

Verifique usando a view que a contagem de alugueis do filme subiu.

In [96]:
db("""
SELECT mv.cnt
FROM movie_count mv
WHERE mv.title = 'COWBOY DOOM'
""")

Executando query:
(9,)


Faça o *rollback* desta alteração, para não modificar a nossa querida base de dados *sakila*.

In [97]:
connection.rollback()

## Tabelas temporárias

Tabelas temporárias podem ser criadas para ajudar nas tarefas de manipulação de dados. Essas tabelas existem apenas pela duração da sessão. Para criar uma tabela temporária, basta adicionar a palavra-chave `TEMPORARY` no momento da criação.

É comum criar tabelas temporárias à partir do resultado de comandos `SELECT`. Por exemplo, a seguinte query permite montar uma tabela temporária com os filmes que duram mais que 3 horas:

In [98]:
db("""
DROP TABLE IF EXISTS long_film
""")

db("""
CREATE TEMPORARY TABLE long_film 
    SELECT
        *
    FROM
        film
    WHERE
        film.length > 180;
""")

Executando query:
0 linhas afetadas.
Executando query:
39 linhas afetadas.


Podemos verificar que a tabela `long_film` agora existe:

In [99]:
db("DESCRIBE long_film")

Executando query:
('film_id', 'smallint unsigned', 'NO', '', '0', 'NULL')
('title', 'varchar(128)', 'NO', '', None, 'NULL')
('description', 'text', 'YES', '', None, 'NULL')
('release_year', 'year', 'YES', '', None, 'NULL')
('language_id', 'tinyint unsigned', 'NO', '', None, 'NULL')
('original_language_id', 'tinyint unsigned', 'YES', '', None, 'NULL')
('rental_duration', 'tinyint unsigned', 'NO', '', '3', 'NULL')
('rental_rate', 'decimal(4,2)', 'NO', '', '4.99', 'NULL')
('length', 'smallint unsigned', 'YES', '', None, 'NULL')
('replacement_cost', 'decimal(5,2)', 'NO', '', '19.99', 'NULL')
('rating', "enum('G','PG','PG-13','R','NC-17')", 'YES', '', 'G', 'NULL')
('special_features', "set('Trailers','Commentaries','Deleted Scenes','Behind the Scenes')", 'YES', '', None, 'NULL')
('last_update', 'timestamp', 'NO', '', 'CURRENT_TIMESTAMP', 'on update CURRENT_TIMESTAMP')


In [100]:
db("CALL sys.table_exists('sakila', 'long_film', @table_type);")
db("SELECT @table_type;")

Executando query:
0 linhas afetadas.
Executando query:
('TEMPORARY',)


Muito embora ela não apareça na lista de tabelas: isso é um bug do MySQL. (https://dev.mysql.com/worklog/task/?id=648)

In [101]:
db("SHOW TABLES")

Executando query:
('actor',)
('actor_info',)
('address',)
('category',)
('city',)
('country',)
('customer',)
('customer_list',)
('film',)
('film_actor',)
('film_category',)
('film_list',)
('film_text',)
('inventory',)
('language',)
('movie_count',)
('nicer_but_slower_film_list',)
('payment',)
('rental',)
('sales_by_film_category',)
('sales_by_store',)
('staff',)
('staff_list',)
('store',)


Vamos listar o conteudo desta tabela:

In [102]:
db("SELECT title FROM long_film")

Executando query:
('ANALYZE HOOSIERS',)
('BAKED CLEOPATRA',)
('CATCH AMISTAD',)
('CHICAGO NORTH',)
('CONSPIRACY SPIRIT',)
('CONTROL ANTHEM',)
('CRYSTAL BREAKING',)
('DARN FORRESTER',)
('FRONTIER CABIN',)
('GANGS PRIDE',)
('HAUNTING PIANIST',)
('HOME PITY',)
('HOTEL HAPPINESS',)
('INTRIGUE WORST',)
('JACKET FRISCO',)
('KING EVOLUTION',)
('LAWLESS VISION',)
('LOVE SUICIDES',)
('MONSOON CAUSE',)
('MOONWALKER FOOL',)
('MUSCLE BRIGHT',)
('POND SEATTLE',)
('RECORDS ZORRO',)
('REDS POCUS',)
('RUNAWAY TENENBAUMS',)
('SATURN NAME',)
('SCALAWAG DUCK',)
('SEARCHERS WAIT',)
('SMOOCHY CONTROL',)
('SOLDIERS EVOLUTION',)
('SONS INTERVIEW',)
('SORORITY QUEEN',)
('STAR OPERATION',)
('SWEET BROTHERHOOD',)
('THEORY MERMAID',)
('WIFE TURN',)
('WILD APOLLO',)
('WORST BANGER',)
('YOUNG LANGUAGE',)


Vamos apagar a tabela `long_film`:

In [103]:
db("DROP TABLE long_film")

Executando query:
0 linhas afetadas.


### Vamos praticar

- Crie uma tabela temporária `max_duration` que contém a duração máxima de filme para cada categoria. Apresente o id da categoria, seu nome e a duração máxima.

**Obs**: Como nome das colunas, utilize (nesta ordem): `category_id`, `name`, `max_len`.

In [104]:
# Executamos o DROP apenas localmente, sem enviar ao servidor
db("DROP TABLE IF EXISTS max_duration")

sql_ex05 = """
CREATE TEMPORARY TABLE max_duration
    SELECT c.category_id, c.name, MAX(f.length) as max_len
    FROM category c
    JOIN film_category fc USING(category_id)
    JOIN film f USING(film_id)
    GROUP BY c.category_id
"""

db(sql_ex05)

Executando query:
0 linhas afetadas.
Executando query:
16 linhas afetadas.


In [105]:
ia.sender(answer="sql_ex05", task="views", question="ex05", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex05', style=ButtonStyle()), Output()), _dom_classes=('widget…

 - Verifique a tabela.

In [106]:
db("SELECT * FROM max_duration")

Executando query:
(1, 'Action', 185)
(2, 'Animation', 185)
(3, 'Children', 178)
(4, 'Classics', 184)
(5, 'Comedy', 185)
(6, 'Documentary', 183)
(7, 'Drama', 181)
(8, 'Family', 184)
(9, 'Foreign', 184)
(10, 'Games', 185)
(11, 'Horror', 181)
(12, 'Music', 185)
(13, 'New', 183)
(14, 'Sci-Fi', 185)
(15, 'Sports', 184)
(16, 'Travel', 185)


In [107]:
db("DESCRIBE max_duration")

Executando query:
('category_id', 'tinyint unsigned', 'NO', '', '0', 'NULL')
('name', 'varchar(25)', 'NO', '', None, 'NULL')
('max_len', 'smallint unsigned', 'YES', '', None, 'NULL')


- Agora use a tabela temporária para construir uma consulta com as categorias e seus respectivos filmes mais longos:

In [108]:
db("""
SELECT
    fc.category_id,
    md.name,
    f.film_id,
    f.title,
    f.length
FROM
    film f
    INNER JOIN film_category fc USING (film_id)
    INNER JOIN max_duration md USING (category_id)
WHERE
    f.length = md.max_len
""")

Executando query:
(1, 'Action', 212, 'DARN FORRESTER', 185)
(1, 'Action', 991, 'WORST BANGER', 185)
(2, 'Animation', 349, 'GANGS PRIDE', 185)
(2, 'Animation', 690, 'POND SEATTLE', 185)
(3, 'Children', 344, 'FURY MURDER', 178)
(3, 'Children', 993, 'WRONG BEHAVIOR', 178)
(4, 'Classics', 180, 'CONSPIRACY SPIRIT', 184)
(5, 'Comedy', 182, 'CONTROL ANTHEM', 185)
(6, 'Documentary', 973, 'WIFE TURN', 183)
(6, 'Documentary', 996, 'YOUNG LANGUAGE', 183)
(7, 'Drama', 473, 'JACKET FRISCO', 181)
(8, 'Family', 499, 'KING EVOLUTION', 184)
(9, 'Foreign', 198, 'CRYSTAL BREAKING', 184)
(9, 'Foreign', 821, 'SORORITY QUEEN', 184)
(10, 'Games', 141, 'CHICAGO NORTH', 185)
(11, 'Horror', 24, 'ANALYZE HOOSIERS', 181)
(11, 'Horror', 535, 'LOVE SUICIDES', 181)
(12, 'Music', 426, 'HOME PITY', 185)
(13, 'New', 340, 'FRONTIER CABIN', 183)
(14, 'Sci-Fi', 817, 'SOLDIERS EVOLUTION', 185)
(15, 'Sports', 813, 'SMOOCHY CONTROL', 184)
(16, 'Travel', 609, 'MUSCLE BRIGHT', 185)
(16, 'Travel', 872, 'SWEET BROTHERHOOD', 185)

- delete a tabela temporária

In [109]:
db("""
DROP TABLE max_duration
""")

Executando query:
0 linhas afetadas.


## Variáveis

Podemos montar uma query que retorne um valor só e armazenar este valor em uma variável, para uso posterior em outras queries. Para isso vamos usar o prefixo '@' para indicar variáveis, e o comando `SELECT ... INTO`.

Exemplo: quais são os filmes "caros" da nossa base sakila? Vamos descobrir quais filmes custam mais que um desvio padrão acima da média de preços de locação.

Primeiro vamos calcular a média e o desvio padrão dos preços de aluguel:

In [110]:
db("""
SELECT 
    AVG(rental_rate), 
    STDDEV(rental_rate)
INTO 
    @avg_rate, 
    @stddev_rate 
FROM
    film;
""")

Executando query:
1 linhas afetadas.


Note que a query não retorna um resultado: o resultado foi armazenado direto nas variáveis `@avg_rate` e `@stddev_rate`. Vamos usar um `SELECT` sem tabelas para ver o resultado:

In [111]:
db("SELECT @avg_rate, @stddev_rate")

Executando query:
(Decimal('2.980000000'), 1.6455698101265719)


Agora podemos selecionar os filmes caros!

In [112]:
db("""
SELECT 
    title, rental_rate
FROM
    film
WHERE
    rental_rate > @avg_rate + @stddev_rate
LIMIT 10
""")

Executando query:
('ACE GOLDFINGER', Decimal('4.99'))
('AIRPLANE SIERRA', Decimal('4.99'))
('AIRPORT POLLOCK', Decimal('4.99'))
('ALADDIN CALENDAR', Decimal('4.99'))
('ALI FOREVER', Decimal('4.99'))
('AMELIE HELLFIGHTERS', Decimal('4.99'))
('AMERICAN CIRCUS', Decimal('4.99'))
('ANTHEM LUKE', Decimal('4.99'))
('APACHE DIVINE', Decimal('4.99'))
('APOCALYPSE FLAMINGOS', Decimal('4.99'))


### Vamos praticar

Armazene na variável temporária `max_films` a quantidade de filmes feitos pelo ator ou atriz que mais participou de filmes.

In [113]:
sql_ex06 = """
SELECT MAX(film_count)
INTO @max_films
FROM (
    SELECT actor_id, COUNT(*) AS film_count
    FROM film_actor
    GROUP BY actor_id
) t;
"""

db(sql_ex06)

Executando query:
1 linhas afetadas.


In [114]:
db("""
SELECT @max_films
""")

Executando query:
(42,)


In [115]:
ia.sender(answer="sql_ex06", task="views", question="ex06", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex06', style=ButtonStyle()), Output()), _dom_classes=('widget…

## Operador `IN`

Suponha que desejamos listar todos os filmes dos 3 atores mais populares. Podemos começar listando os 3 atores mais populares:

In [116]:
db("""
SELECT 
    actor_id, first_name, last_name, COUNT(film_id) AS num_films
FROM
    actor
    INNER JOIN film_actor USING (actor_id)
GROUP BY 
    actor_id
ORDER BY 
    num_films DESC
LIMIT 3
""")

Executando query:
(107, 'GINA', 'DEGENERES', 42)
(102, 'WALTER', 'TORN', 41)
(198, 'MARY', 'KEITEL', 40)


Vamos criar uma tabela temporária para guardar a informação de `actor_id` desses atores:

In [117]:
db("DROP TABLE IF EXISTS temp_pop_actors")
db("""
CREATE TEMPORARY TABLE temp_pop_actors
    SELECT first_name, last_name, actor_id FROM
        actor
        INNER JOIN film_actor USING (actor_id)
    GROUP BY 
        actor_id
    ORDER BY 
        COUNT(film_id) DESC
    LIMIT 3
""")
db("""
SELECT * from temp_pop_actors
""")

Executando query:
0 linhas afetadas.
Executando query:
3 linhas afetadas.
Executando query:
('GINA', 'DEGENERES', 107)
('WALTER', 'TORN', 102)
('MARY', 'KEITEL', 198)


Por fim, vamos usar essa informação para listar os filmes dos atores populares:

In [118]:
db("""
SELECT DISTINCT
    title
FROM
    film
    INNER JOIN film_actor USING (film_id)
WHERE
    actor_id IN (SELECT actor_id FROM temp_pop_actors);
""")

Executando query:
('BED HIGHBALL',)
('CALENDAR GUNFIGHT',)
('CHAMBER ITALIAN',)
('CHAPLIN LICENSE',)
('CHARIOTS CONSPIRACY',)
('CLUELESS BUCKET',)
('COLDBLOODED DARLING',)
('CONEHEADS SMOOCHY',)
('DARKNESS WAR',)
('DEER VIRGINIAN',)
('DOGMA FAMILY',)
('ELEPHANT TROJAN',)
('EXCITEMENT EVE',)
('FRISCO FORREST',)
('GANDHI KWAI',)
('GOODFELLAS SALUTE',)
('GUNFIGHT MOON',)
('HALL CASSIDY',)
('HEARTBREAKERS BRIGHT',)
('HOOK CHARIOTS',)
('HYDE DOCTOR',)
('IMPACT ALADDIN',)
('INDIAN LOVE',)
('INTRIGUE WORST',)
('LICENSE WEEKEND',)
('LOUISIANA HARRY',)
('MAGNIFICENT CHITTY',)
('METAL ARMAGEDDON',)
('MIDNIGHT WESTWARD',)
('MOVIE SHAKESPEARE',)
('MUMMY CREATURES',)
('OPEN AFRICAN',)
('SEARCHERS WAIT',)
('SEVEN SWARM',)
('SIERRA DIVIDE',)
('SPIRITED CASUALTIES',)
('STORM HAPPINESS',)
('SUGAR WONKA',)
('TELEGRAPH VOYAGE',)
('TRAINSPOTTING STRANGERS',)
('WIFE TURN',)
('WINDOW SIDE',)
('AMELIE HELLFIGHTERS',)
('ARABIA DOGMA',)
('BANG KWAI',)
('CASABLANCA SUPER',)
('CASPER DRAGONFLY',)
('CROW GREASE',

Note o uso de *subqueries*!

Não se esqueça de limpar tudo no final!

In [119]:
db("DROP TABLE temp_pop_actors")

Executando query:
0 linhas afetadas.


### Vamos praticar

Liste os atores (id, nome e sobrenome) que participaram dos 3 filmes mais rentáveis (aqueles que mais geraram receita para a locadora) ordenados pelo id do ator de modo crescente. Para isso, crie uma tabela temporária contendo o id do filme e a quantia total e use essa tabela para listar os atores.

In [120]:
db("""
CREATE TEMPORARY TABLE temp
    SELECT f.film_id, SUM(p.amount)
    FROM film f
    JOIN inventory i USING (film_id)
    JOIN rental r USING(inventory_id)
    JOIN payment p USING(rental_id)
    GROUP BY f.film_id
   ORDER BY SUM(p.amount) DESC
   LIMIT 3
""")
db("""
SELECT DISTINCT
    a.actor_id, a.first_name, a.last_name
FROM
    actor a
        JOIN
    film_actor fa USING (actor_id)
        JOIN
    temp t USING (film_id)
ORDER BY actor_id ASC
""")

Executando query:
3 linhas afetadas.
Executando query:
(28, 'WOODY', 'HOFFMAN')
(47, 'JULIA', 'BARRYMORE')
(52, 'CARMEN', 'HUNT')
(107, 'GINA', 'DEGENERES')
(111, 'CAMERON', 'ZELLWEGER')
(138, 'LUCILLE', 'DEE')
(155, 'IAN', 'TANDY')
(157, 'GRETA', 'MALDEN')
(158, 'VIVIEN', 'BASINGER')
(166, 'NICK', 'DEGENERES')
(174, 'MICHAEL', 'BENING')
(178, 'LISA', 'MONROE')
(185, 'MICHAEL', 'BOLGER')
(200, 'THORA', 'TEMPLE')


In [121]:
sql_ex07 = ["""
CREATE TEMPORARY TABLE temp
    SELECT f.film_id, SUM(p.amount)
    FROM film f
    JOIN inventory i USING (film_id)
    JOIN rental r USING(inventory_id)
    JOIN payment p USING(rental_id)
    GROUP BY f.film_id
   ORDER BY SUM(p.amount) DESC
   LIMIT 3
""",
"""
SELECT DISTINCT
    a.actor_id, a.first_name, a.last_name
FROM
    actor a
        JOIN
    film_actor fa USING (actor_id)
        JOIN
    temp t USING (film_id)
ORDER BY actor_id ASC
"""]

ia.sender(answer="sql_ex07", task="views", question="ex07", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex07', style=ButtonStyle()), Output()), _dom_classes=('widget…

## Subqueries

Os tópicos discutidos acima poderiam ser resolvidos, em grande parte, com subqueries. As subqueries são queries `SELECT` criadas dentro de outras queries. 

Poderíamos ter usado subqueries nos mesmos lugares onde usamos tabelas temporárias, nos tópicos acima. Quando a subquery pode ser transformada em uma tabela temporária independente, separada da query exterior, dizemos que a subquery é **não-correlacionada** com a query exterior.

Usar subqueries não-correlacionadas é um tópico controverso: podemos sempre usar uma tabela temporária ou, ás vezes, pensar em um `JOIN` simples. Aliás, muitas vezes o otimizador de queries do banco de dados transformará a subquery em `JOIN`, se isso for vantajoso em termos de desempenho.

Uma subquery que depende da query externa (e portanto não pode ser separada em uma tabela temporária independente) é chamada de **subquery correlacionada**. Nestes casos podemos ter que executar a subquery para cada linha da query exterior! 

### Exemplo 1 (subquery)

Query para retornar o autor e a quantidade de músicas vinculadas a cada autor.

```sql
USE musica;
```

- Versão **sem subquery**:

```sql
SELECT 
    a.Nome_Autor, COUNT(*) qtde_musicas
FROM
    AUTOR a
        INNER JOIN
    MUSICA_AUTOR ma USING (Codigo_Autor)
GROUP BY Codigo_Autor;
```

- Versão **com subquery**:

```sql
SELECT 
    a.Nome_Autor,
    (SELECT 
            COUNT(*)
        FROM
            MUSICA_AUTOR ma
        WHERE
            ma.Codigo_Autor = a.Codigo_Autor) qtde_musicas
FROM
    AUTOR a;
```

### Exemplo 2 (subquery)

Query para retornar o autor e a média de duração de suas músicas, apenas para as músicas que possuem `a` em seu título e, então, filtrar apenas os autores com duração média entre `3.5` e `3.8`.

```sql
USE musica;
```

- Versão **sem subquery** usando `HAVING`:

```sql
SELECT 
    a.Nome_Autor AS nome, AVG(m.Duracao) AS duracao_media
FROM
    MUSICA AS m
        INNER JOIN
    MUSICA_AUTOR AS ma USING (Codigo_Musica)
        INNER JOIN
    AUTOR AS a USING (Codigo_Autor)
WHERE
    m.Nome_Musica LIKE '%a%'
GROUP BY a.Codigo_Autor
HAVING AVG(m.Duracao) BETWEEN 3.5 AND 3.8
ORDER BY duracao_media DESC;
```

- Versão **com subquery**:

```sql
SELECT 
    t.nome, t.duracao_media
FROM -- Vamos utilizar subquery
    (SELECT 
        a.Nome_Autor AS nome, AVG(m.Duracao) AS duracao_media
    FROM
        MUSICA AS m
    INNER JOIN MUSICA_AUTOR AS ma USING (Codigo_Musica)
    INNER JOIN AUTOR AS a USING (Codigo_Autor)
    WHERE
        m.Nome_Musica LIKE '%a%'
    GROUP BY a.Codigo_Autor
    ORDER BY duracao_media DESC) AS t -- Vamos criar um alias para a nossa subquery
WHERE
    t.duracao_media BETWEEN 3.5 AND 3.8;
```

### Vamos praticar

Vamos refazer a atividade dos atores dos filmes mais rentáveis, usando subqueries. 

Temos um problema: o MySQL não suporta ``LIMIT`` em subqueries com o operador ``IN``. Vamos investigar isso mais de perto. 

Em primeiro lugar faça uma tradução direta da implementação da atividade anterior trocando tabela temporária por subquery.

In [122]:
try:
    db("""
    SELECT DISTINCT
    a.actor_id, a.first_name, a.last_name
FROM actor a
JOIN film_actor fa USING (actor_id)
WHERE fa.film_id IN (
    SELECT film_id FROM (
        SELECT f.film_id
        FROM film f
        JOIN inventory i USING (film_id)
        JOIN rental r USING (inventory_id)
        JOIN payment p USING (rental_id)
        GROUP BY f.film_id
        ORDER BY SUM(p.amount) DESC
        LIMIT 3
    ) AS top_films
)
ORDER BY a.actor_id ASC;
    """)
except mysql.connector.ProgrammingError as e:
    print(f"ProgrammingError: {e}")

Executando query:
(28, 'WOODY', 'HOFFMAN')
(47, 'JULIA', 'BARRYMORE')
(52, 'CARMEN', 'HUNT')
(107, 'GINA', 'DEGENERES')
(111, 'CAMERON', 'ZELLWEGER')
(138, 'LUCILLE', 'DEE')
(155, 'IAN', 'TANDY')
(157, 'GRETA', 'MALDEN')
(158, 'VIVIEN', 'BASINGER')
(166, 'NICK', 'DEGENERES')
(174, 'MICHAEL', 'BENING')
(178, 'LISA', 'MONROE')
(185, 'MICHAEL', 'BOLGER')
(200, 'THORA', 'TEMPLE')


Ok, apareceu o problema. Mas considere que o problema original não precisava de IN desde o começo! Construa essa solução.

In [123]:
sql_ex08 = """
SELECT DISTINCT
    a.actor_id, a.first_name, a.last_name
FROM actor a
JOIN film_actor fa USING (actor_id)
WHERE fa.film_id IN (
    SELECT film_id FROM (
        SELECT f.film_id
        FROM film f
        JOIN inventory i USING (film_id)
        JOIN rental r USING (inventory_id)
        JOIN payment p USING (rental_id)
        GROUP BY f.film_id
        ORDER BY SUM(p.amount) DESC
        LIMIT 3
    ) AS top_films
)
ORDER BY a.actor_id ASC;
"""

db(sql_ex08)

Executando query:
(28, 'WOODY', 'HOFFMAN')
(47, 'JULIA', 'BARRYMORE')
(52, 'CARMEN', 'HUNT')
(107, 'GINA', 'DEGENERES')
(111, 'CAMERON', 'ZELLWEGER')
(138, 'LUCILLE', 'DEE')
(155, 'IAN', 'TANDY')
(157, 'GRETA', 'MALDEN')
(158, 'VIVIEN', 'BASINGER')
(166, 'NICK', 'DEGENERES')
(174, 'MICHAEL', 'BENING')
(178, 'LISA', 'MONROE')
(185, 'MICHAEL', 'BOLGER')
(200, 'THORA', 'TEMPLE')


In [124]:
ia.sender(answer="sql_ex08", task="views", question="ex08", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex08', style=ButtonStyle()), Output()), _dom_classes=('widget…

# `UNION`

Quando duas tabelas tem **EXATAMENTE** as mesmas colunas, podemos concatená-las e formar uma grande tabela unificada usando o operador `UNION`. Por exemplo: suponha que desejamos montar uma lista dos nomes e sobrenomes de todos os clientes E de todos os funcionários. Eis uma solução possível:

In [125]:
db("DROP TABLE IF EXISTS nomes_clientes")
db("""
CREATE TEMPORARY TABLE nomes_clientes 
    SELECT first_name, last_name FROM customer
""")

Executando query:
0 linhas afetadas.
Executando query:
599 linhas afetadas.


In [126]:
db("DESCRIBE nomes_clientes")
db("SELECT * FROM nomes_clientes LIMIT 5")

Executando query:
('first_name', 'varchar(45)', 'NO', '', None, 'NULL')
('last_name', 'varchar(45)', 'NO', '', None, 'NULL')
Executando query:
('MARY', 'SMITH')
('PATRICIA', 'JOHNSON')
('LINDA', 'WILLIAMS')
('BARBARA', 'JONES')
('ELIZABETH', 'BROWN')


In [127]:
db("DROP TABLE IF EXISTS nomes_staff")
db("""
CREATE TEMPORARY TABLE nomes_staff 
    SELECT first_name, last_name FROM staff
""")

Executando query:
0 linhas afetadas.
Executando query:
2 linhas afetadas.


In [128]:
db("DESCRIBE nomes_staff")
db("SELECT * FROM nomes_staff LIMIT 5")

Executando query:
('first_name', 'varchar(45)', 'NO', '', None, 'NULL')
('last_name', 'varchar(45)', 'NO', '', None, 'NULL')
Executando query:
('Mike', 'Hillyer')
('Jon', 'Stephens')


In [129]:
db("DROP TABLE IF EXISTS nomes_all")
db("""
CREATE TEMPORARY TABLE nomes_all 
    (SELECT * FROM nomes_staff)
    UNION 
    (SELECT * FROM nomes_clientes)
""")

Executando query:
0 linhas afetadas.
Executando query:
601 linhas afetadas.


In [130]:
db("DESCRIBE nomes_all")
db("SELECT * FROM nomes_all LIMIT 5")

Executando query:
('first_name', 'varchar(45)', 'NO', '', '', 'NULL')
('last_name', 'varchar(45)', 'NO', '', '', 'NULL')
Executando query:
('Mike', 'Hillyer')
('Jon', 'Stephens')
('MARY', 'SMITH')
('PATRICIA', 'JOHNSON')
('LINDA', 'WILLIAMS')


In [131]:
db("DROP TABLE IF EXISTS nomes_clientes")
db("DROP TABLE IF EXISTS nomes_staff")
db("DROP TABLE IF EXISTS nomes_all")

Executando query:
0 linhas afetadas.
Executando query:
0 linhas afetadas.
Executando query:
0 linhas afetadas.


**Vamos praticar:** refaça o exemplo acima mas use *subqueries* ao invés de *temp tables*. Ordene de forma ascendente por first_name e last_name. Também é possível resolver apenas com união de `SELECT`s, sem *subquery*.

In [132]:
sql_ex09 = """
( SELECT first_name, last_name FROM customer) UNION (SELECT first_name, last_name FROM staff)
ORDER BY first_name ASC, last_name ASC
"""
db(sql_ex09)

Executando query:
('AARON', 'SELBY')
('ADAM', 'GOOCH')
('ADRIAN', 'CLARY')
('AGNES', 'BISHOP')
('ALAN', 'KAHN')
('ALBERT', 'CROUSE')
('ALBERTO', 'HENNING')
('ALEX', 'GRESHAM')
('ALEXANDER', 'FENNELL')
('ALFRED', 'CASILLAS')
('ALFREDO', 'MCADAMS')
('ALICE', 'STEWART')
('ALICIA', 'MILLS')
('ALLAN', 'CORNISH')
('ALLEN', 'BUTTERFIELD')
('ALLISON', 'STANLEY')
('ALMA', 'AUSTIN')
('ALVIN', 'DELOACH')
('AMANDA', 'CARTER')
('AMBER', 'DIXON')
('AMY', 'LOPEZ')
('ANA', 'BRADLEY')
('ANDRE', 'RAPP')
('ANDREA', 'HENDERSON')
('ANDREW', 'PURDY')
('ANDY', 'VANHORN')
('ANGEL', 'BARCLAY')
('ANGELA', 'HERNANDEZ')
('ANITA', 'MORALES')
('ANN', 'EVANS')
('ANNA', 'HILL')
('ANNE', 'POWELL')
('ANNETTE', 'OLSON')
('ANNIE', 'RUSSELL')
('ANTHONY', 'SCHWAB')
('ANTONIO', 'MEEK')
('APRIL', 'BURNS')
('ARLENE', 'HARVEY')
('ARMANDO', 'GRUBER')
('ARNOLD', 'HAVENS')
('ARTHUR', 'SIMPKINS')
('ASHLEY', 'RICHARDSON')
('AUDREY', 'RAY')
('AUSTIN', 'CINTRON')
('BARBARA', 'JONES')
('BARRY', 'LOVELACE')
('BEATRICE', 'ARNOLD')
('BEC

In [133]:
ia.sender(answer="sql_ex09", task="views", question="ex09", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex09', style=ButtonStyle()), Output()), _dom_classes=('widget…

## Desafios!

Faça uma lista de filmes que tenham mais de dois atores cujo nome inicia com a mesma letra do título do filme! Apresente o nome e a quantidade de atores. Ordene pelo id do filme. Dica: Pesquise a função `LEFT`

In [134]:
sql_ex10 = """
SELECT f.title, COUNT(sub.actor_id) as contador_ator
FROM film f
JOIN(
SELECT a.actor_id, f.film_id
FROM actor a
JOIN film_actor fa USING(actor_id)
JOIN film f USING(film_id)
WHERE LEFT(a.first_name, 1) = LEFT(f.title, 1)
)AS sub USING(film_id)
GROUP BY f.film_id
HAVING contador_ator > 2
ORDER BY f.film_id ASC
"""
db(sql_ex10)

Executando query:
('CROW GREASE', 3)
('JEDI BENEATH', 3)
('SUBMARINE BED', 3)


In [135]:
ia.sender(answer="sql_ex10", task="views", question="ex10", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex10', style=ButtonStyle()), Output()), _dom_classes=('widget…

Semana do "DAN HARRIS": liste os clientes (nome e sobrenome) que nunca assistiram um filme do ator "DAN HARRIS" ou que já assistiram mas a ultima vez em que assistiram um filme dele foi antes de '2005-06-01'. Ordene pelo nome e sobrenome do cliente.

Considerem que a pessoa pode ter assistido um filme de Dan Harris se:

- Devolveu em 2005-06-01 ou depois
- Alugou em 2005-06-01 ou depois
- Alugou antes de 2005-06-01 e ainda não devolveu

In [136]:
sql_ex11 = """
SELECT c1.first_name, c1.last_name
FROM customer c1
WHERE c1.customer_id NOT IN(
SELECT 
    c.customer_id
FROM
    customer c
        LEFT JOIN
    rental r USING (customer_id)
        JOIN
    inventory i USING (inventory_id)
        JOIN
    film f USING (film_id)
        JOIN
    film_actor fa USING (film_id)
        JOIN
    actor a USING (actor_id)
WHERE
    a.first_name = 'DAN'
        AND a.last_name = 'HARRIS'
        AND (r.return_date >= '2005-06-01'
        OR r.rental_date >= '2005-06-01'
        OR (r.rental_date < '2005-06-01'
        AND r.return_date IS NULL))
)
ORDER BY c1.first_name ASC, c1.last_name ASC
"""
db(sql_ex11)

Executando query:
('AARON', 'SELBY')
('AGNES', 'BISHOP')
('ALAN', 'KAHN')
('ALBERT', 'CROUSE')
('ALICIA', 'MILLS')
('ALLAN', 'CORNISH')
('ALLEN', 'BUTTERFIELD')
('ALLISON', 'STANLEY')
('AMANDA', 'CARTER')
('AMY', 'LOPEZ')
('ANA', 'BRADLEY')
('ANDREA', 'HENDERSON')
('ANDREW', 'PURDY')
('ANGEL', 'BARCLAY')
('ANITA', 'MORALES')
('ANN', 'EVANS')
('ANNA', 'HILL')
('ANNE', 'POWELL')
('ANNETTE', 'OLSON')
('ANNIE', 'RUSSELL')
('ANTHONY', 'SCHWAB')
('ANTONIO', 'MEEK')
('ARMANDO', 'GRUBER')
('AUDREY', 'RAY')
('AUSTIN', 'CINTRON')
('BEATRICE', 'ARNOLD')
('BECKY', 'MILES')
('BEN', 'EASTER')
('BERNARD', 'COLBY')
('BESSIE', 'MORRISON')
('BETH', 'FRANKLIN')
('BILLIE', 'HORTON')
('BILLY', 'POULIN')
('BOB', 'PFEIFFER')
('BRAD', 'MCCURDY')
('BRANDY', 'GRAVES')
('BRENT', 'HARKINS')
('BRETT', 'CORNWELL')
('BRIAN', 'WYMAN')
('BRITTANY', 'RILEY')
('BYRON', 'BOX')
('CALVIN', 'MARTEL')
('CARLA', 'GUTIERREZ')
('CAROLE', 'BARNETT')
('CARRIE', 'PORTER')
('CASEY', 'MENA')
('CATHY', 'SPENCER')
('CECIL', 'VINES')
(

In [146]:
ia.sender(answer="sql_ex11", task="views", question="ex11", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex11', style=ButtonStyle()), Output()), _dom_classes=('widget…

- Faça uma consulta que retorne, para cada ator, a seguinte informação:

| first_name | last_name | filmes por categoria |
|--|--|--|
| PENELOPE | GUINESS | Animation: ANACONDA CONFESSIONS; Children: LANGUAGE COWBOY; Classics: COLOR PHILADELPHIA, WESTWARD SEABISCUIT; Comedy: VERTIGO NORTHWEST; Documentary: ACADEMY DINOSAUR; Family: KING EVOLUTION, SPLASH GUMP; Foreign: MULHOLLAND BEAST; Games: BULWORTH COMMANDMENTS, HUMAN GRAFFITI; Horror: ELEPHANT TROJAN, LADY STAGE, RULES HUMAN; Music: WIZARD COLDBLOODED; New: ANGELS LIFE, OKLAHOMA JUMANJI; Sci-Fi: CHEAPER CLYDE; Sports: GLEAMING JAWBREAKER |
| NICK | WAHLBERG | Action: BULL SHAWSHANK; Animation: FIGHT JAWBREAKER; Children: JERSEY SASSY; Classics: DRACULA CRYSTAL, GILBERT PELICAN; Comedy: MALLRATS UNITED, RUSHMORE MERMAID; Documentary: ADAPTATION HOLES; Drama: WARDROBE PHANTOM; Family: APACHE DIVINE, CHISUM BEHAVIOR, INDIAN LOVE, MAGUIRE APACHE; Foreign: BABY HALL, HAPPINESS UNITED; Games: ROOF CHAMPION; Music: LUCKY FLYING; New: DESTINY SATURDAY, FLASH WARS, JEKYLL FROGMEN, MASK PEACH; Sci-Fi: CHAINSAW UPTOWN, GOODFELLAS SALUTE; Travel: LIAISONS SWEET, SMILE EARRING |
| etc | etc | etc |

Ordene pelo nome e sobrenome do ator.

Dica: use `GROUP_CONCAT` para agrupar todas as strings de uma coluna em uma string só, e `CONCAT` para unir strings particulares.

In [150]:
sql_ex12 = """
SELECT 
    a.first_name,
    a.last_name,
    GROUP_CONCAT(
        CONCAT(sub.name, ': ', sub.films_by_category)
        SEPARATOR '; '
    ) AS filmes_por_categoria
FROM actor a
JOIN (
    SELECT 
        fa.actor_id,
        c.name,
        GROUP_CONCAT(f.title ORDER BY f.title SEPARATOR ', ') AS films_by_category
    FROM film_actor fa
    JOIN film f USING (film_id)
    JOIN film_category fc USING (film_id)
    JOIN category c USING (category_id)
    GROUP BY fa.actor_id, c.name
) sub ON a.actor_id = sub.actor_id
GROUP BY a.actor_id, a.first_name, a.last_name
ORDER BY a.first_name, a.last_name;
"""

db(sql_ex12)

OperationalError: MySQL Connection not available.

In [149]:
ia.sender(answer="sql_ex12", task="views", question="ex12", answer_type="pyvar")

interactive(children=(Button(description='Enviar ex12', style=ButtonStyle()), Output()), _dom_classes=('widget…

## Conclusão

Façamos uma pausa para apreciar quão longe estamos: já conseguimos criar nossas tabelas, inserir informação, removê-la, atualizá-la, e consultar nossa base de maneiras bem sofisticadas! Vimos desde `SELECT` simples até buscas mais complexas envolvendo várias etapas de processamento para obter o dado desejado.

Por hoje é só, feche sua conexão e bom descanso!

In [151]:
connection.close()

## Conferir Notas

Confira se as notas na atividade são as esperadas!

Primeiro na atividade atual!

In [152]:
ia.grades(by="TASK", task="views")

|    | Tarefa   |   Nota | Conta como ATV?   |
|---:|:---------|-------:|:------------------|
|  0 | views    |     10 | Sim               |

In [153]:
ia.grades(task="views")

|    | Atividade   | Exercício   |   Peso |   Nota |   Nota Sem Atraso |   Nota Com Atraso |
|---:|:------------|:------------|-------:|-------:|------------------:|------------------:|
|  0 | views       | ex01        |      1 |     10 |                10 |                 0 |
|  1 | views       | ex02        |      1 |     10 |                10 |                 0 |
|  2 | views       | ex03        |      1 |     10 |                10 |                 0 |
|  3 | views       | ex04        |      1 |     10 |                10 |                 0 |
|  4 | views       | ex05        |      1 |     10 |                10 |                 0 |
|  5 | views       | ex06        |      1 |     10 |                10 |                 0 |
|  6 | views       | ex07        |      1 |     10 |                10 |                 0 |
|  7 | views       | ex08        |      1 |     10 |                10 |                 0 |
|  8 | views       | ex09        |      1 |     10 |                10 |                 0 |
|  9 | views       | ex10        |      1 |     10 |                10 |                 0 |
| 10 | views       | ex11        |      1 |     10 |                10 |                 0 |
| 11 | views       | ex12        |      1 |     10 |                10 |                 0 |

In [154]:
ia.grades(by="task")

|    | Tarefa       |   Nota | Conta como ATV?   |
|---:|:-------------|-------:|:------------------|
|  0 | newborn      |     10 | Não               |
|  1 | select01     |     10 | Sim               |
|  2 | ddl          |     10 | Sim               |
|  3 | dml          |     10 | Sim               |
|  4 | agg_join     |     10 | Sim               |
|  5 | group_having |     10 | Sim               |
|  6 | views        |     10 | Sim               |
|  7 | sql_review1  |      0 | Sim               |

Média de ATV:

In [155]:
# Média de ATV, dividindo por n-2
ia.average(excluded_count=2)

|    |   Média de ATV |
|---:|---------------:|
|  0 |             10 |